# Week 2 Project — OBD-II Vehicle Data Analysis

This notebook combines the main topics of Week 2 using the OBD-II vehicle dataset:

- Descriptive statistics
- Probability
- Linear algebra
- Exploratory data analysis
- Outlier detection
- Correlation and data storytelling

## Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../OBDii_data.csv", low_memory=False)

print(df.shape)
df.head()

## Inspect and clean the data

In [ ]:
df.info()

In [ ]:
print("Empty rows:", df.isna().all(axis=1).sum())
print("Empty columns:", df.isna().all(axis=0).sum())
print("Duplicated rows:", df.duplicated().sum())

df.dropna(how="all", inplace=True)
df.dropna(how="all", axis=1, inplace=True)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print("Shape after cleaning:", df.shape)

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

Some sensors contain missing values because not every vehicle reports all OBD-II measurements. For this reason, each analysis uses the available values from the required columns.

## Select the main vehicle features

In [ ]:
cars = df[
    [
        "ENGINE_RPM",
        "SPEED",
        "ENGINE_COOLANT_TEMP",
        "AIR_INTAKE_TEMP",
        "BAROMETRIC_PRESSURE(KPA)",
        "INTAKE_MANIFOLD_PRESSURE"
    ]
].copy()

cars.head()

## Descriptive statistics

In [ ]:
cars.describe().T

### Speed statistics

In [ ]:
speed = cars["SPEED"].dropna()

mean_speed = speed.mean()
median_speed = speed.median()
mode_speed = speed.mode()[0]
speed_range = speed.max() - speed.min()
speed_variance = speed.var()
speed_std = speed.std()

q1 = speed.quantile(0.25)
q3 = speed.quantile(0.75)
iqr = q3 - q1

print("Mean:", round(mean_speed, 2))
print("Median:", round(median_speed, 2))
print("Mode:", round(mode_speed, 2))
print("Range:", round(speed_range, 2))
print("Variance:", round(speed_variance, 2))
print("Standard deviation:", round(speed_std, 2))
print("Q1:", round(q1, 2))
print("Q3:", round(q3, 2))
print("IQR:", round(iqr, 2))

The speed statistics describe the center and spread of the observations. The mean and median show the typical speed, while the standard deviation and IQR show how much the values vary.

## Probability using vehicle readings

In [ ]:
probability_data = df[
    ["SPEED", "ENGINE_RPM"]
].dropna()

high_speed = probability_data["SPEED"] > 60
high_rpm = probability_data["ENGINE_RPM"] > 2500

p_high_speed = high_speed.mean()
p_high_rpm = high_rpm.mean()
p_both = (high_speed & high_rpm).mean()

p_high_speed_given_high_rpm = p_both / p_high_rpm

print("P(Speed > 60):", round(p_high_speed, 3))
print("P(RPM > 2500):", round(p_high_rpm, 3))
print("P(Speed > 60 and RPM > 2500):", round(p_both, 3))
print(
    "P(Speed > 60 | RPM > 2500):",
    round(p_high_speed_given_high_rpm, 3)
)

This section applies experimental probability to real observations. The conditional probability shows how often high speed occurs when the engine RPM is also high.

## Linear algebra

In [ ]:
features = [
    "ENGINE_RPM",
    "SPEED",
    "ENGINE_COOLANT_TEMP"
]

samples = df[features].dropna().head(3).to_numpy()

weights = np.array([0.003, 0.5, 0.2])

print("Samples:")
print(samples)
print("Shape:", samples.shape)

first_prediction = np.dot(samples[0], weights)
predictions = samples @ weights

print("First prediction:", round(first_prediction, 2))
print("All predictions:", predictions.round(2))

Each row is represented as a feature vector. The dot product gives one result for the first sample, while matrix multiplication calculates a result for all three samples at once.

These values are only examples of how a linear model combines features and weights; they are not trained predictions.

## Feature distributions

In [ ]:
cars.hist(figsize=(12, 9), bins=25)
plt.suptitle("Distribution of Selected Vehicle Features")
plt.tight_layout()
plt.show()

The histograms show that the selected vehicle measurements have different ranges and distribution shapes. Speed and engine RPM have wider variation than most temperature and pressure features.

## Box plots

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(data=cars, x="ENGINE_RPM")
plt.title("Engine RPM Distribution")
plt.xlabel("Engine RPM")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
sns.boxplot(data=cars, x="SPEED")
plt.title("Vehicle Speed Distribution")
plt.xlabel("Speed")
plt.tight_layout()
plt.show()

## Detect RPM outliers using IQR

In [ ]:
rpm = cars["ENGINE_RPM"].dropna()

rpm_q1 = rpm.quantile(0.25)
rpm_q3 = rpm.quantile(0.75)
rpm_iqr = rpm_q3 - rpm_q1

lower_bound = rpm_q1 - 1.5 * rpm_iqr
upper_bound = rpm_q3 + 1.5 * rpm_iqr

rpm_outliers = rpm[
    (rpm < lower_bound) |
    (rpm > upper_bound)
]

outlier_percentage = len(rpm_outliers) / len(rpm) * 100

print("Q1:", round(rpm_q1, 2))
print("Q3:", round(rpm_q3, 2))
print("IQR:", round(rpm_iqr, 2))
print("Lower bound:", round(lower_bound, 2))
print("Upper bound:", round(upper_bound, 2))
print("Number of outliers:", len(rpm_outliers))
print("Outlier percentage:", round(outlier_percentage, 2), "%")

The RPM values outside the IQR bounds are considered potential outliers. They are kept because high RPM may represent real acceleration or high engine load rather than an incorrect reading.

## Engine RPM and speed

In [ ]:
rpm_speed = df[
    ["ENGINE_RPM", "SPEED"]
].dropna()

rpm_speed_corr = rpm_speed.corr().loc[
    "ENGINE_RPM",
    "SPEED"
]

print(
    "Correlation between RPM and speed:",
    round(rpm_speed_corr, 2)
)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=rpm_speed,
    x="SPEED",
    y="ENGINE_RPM",
    alpha=0.4
)

plt.title("Engine RPM vs Vehicle Speed")
plt.xlabel("Speed")
plt.ylabel("Engine RPM")
plt.tight_layout()
plt.show()

Engine RPM and vehicle speed show a clear positive relationship. The relationship is not perfect because gear selection, traffic, road conditions, and driving behavior also affect the readings.

## Speed comparison between vehicle models

In [ ]:
top_models = df["MODEL"].value_counts().head(6).index

model_speed = df[
    df["MODEL"].isin(top_models)
][["MODEL", "SPEED"]].dropna()

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=model_speed,
    x="SPEED",
    y="MODEL"
)

plt.title("Speed Distribution for Common Vehicle Models")
plt.xlabel("Speed")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

The speed distributions are different between some vehicle models. These differences may also be affected by the route, traffic, and number of readings collected from each vehicle.

## Correlation matrix

In [ ]:
corr = cars.corr()

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
corr["SPEED"].sort_values(ascending=False)

The correlation matrix helps identify the features that move together. Engine RPM has a strong positive relationship with speed, while some temperature and pressure features have weaker relationships.

Correlation describes a relationship between variables, but it does not prove that one variable causes the other.

## Pairplot

In [ ]:
pair_data = df[
    [
        "ENGINE_RPM",
        "SPEED",
        "BAROMETRIC_PRESSURE(KPA)",
        "ENGINE_COOLANT_TEMP",
        "AIR_INTAKE_TEMP"
    ]
].dropna()

pair_data = pair_data.sample(
    min(1000, len(pair_data)),
    random_state=40
)

sns.pairplot(
    pair_data,
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.4}
)

plt.show()

The pairplot confirms the relationship between RPM and speed and makes it easier to compare the selected vehicle measurements together.

## Final conclusion

The OBD-II dataset contains useful information about vehicle speed, engine RPM, temperature, and pressure.

The project applied descriptive statistics, probability, vectors and matrices, outlier detection, bivariate analysis, and correlation analysis to the same vehicle dataset.

The strongest visible relationship is between engine RPM and vehicle speed. Some RPM values were detected as potential outliers, but they were kept because they may represent real driving conditions.

Before using this data in a machine learning model, missing values should be handled carefully, features may need scaling, and highly related measurements should be reviewed.